# AccessApp: On-Device ML Verification (MediaPipe)

This Google Colab notebook is provided to verify the core Machine Learning models used in **AccessApp**, satisfying the hackathon ML domain requirements.

Because our application runs exclusively on-device (Zero-Cloud architecture) using Edge TPU optimized models, this notebook replicates the logic using the Python equivalent of our Android MediaPipe pipeline.

## Part 1: Obstacle Radar (Object Detection)
It uses the exact same `efficientdet_lite0.tflite` model (INT8 Quantized) used in our Android APK to detect cars, people, and obstacles in real-time.

In [ ]:
!pip install -q mediapipe opencv-python matplotlib urllib3

In [ ]:
import cv2
import urllib.request
import numpy as np
import matplotlib.pyplot as plt
import mediapipe as mp

# Download the EfficientDet-Lite0 model (Same INT8 model as Android app)
urllib.request.urlretrieve('https://storage.googleapis.com/mediapipe-models/object_detector/efficientdet_lite0/int8/1/efficientdet_lite0.tflite', 'efficientdet_lite0.tflite')

# Download a sample image (Classic computer vision test image with a dog, bicycle, and car)
urllib.request.urlretrieve('https://raw.githubusercontent.com/pjreddie/darknet/master/data/dog.jpg', 'sample.jpg')

print("Radar Model and sample image downloaded successfully.")

In [ ]:
# Initialize the Object Detector
base_options = mp.tasks.BaseOptions(model_asset_path='efficientdet_lite0.tflite')
options = mp.tasks.vision.ObjectDetectorOptions(base_options=base_options, score_threshold=0.3)
detector = mp.tasks.vision.ObjectDetector.create_from_options(options)

# Run Inference 
image = mp.Image.create_from_file('sample.jpg')
detection_result = detector.detect(image)

# Visualize the results (Bounding boxes trigger the haptic feedback in the app)
image_copy = np.copy(image.numpy_view())
for detection in detection_result.detections:
    bbox = detection.bounding_box
    start_point = bbox.origin_x, bbox.origin_y
    end_point = bbox.origin_x + bbox.width, bbox.origin_y + bbox.height
    cv2.rectangle(image_copy, start_point, end_point, (0, 255, 0), 3)
    
    category = detection.categories[0]
    category_name = category.category_name
    probability = round(category.score, 2)
    result_text = category_name + ' (' + str(probability) + ')'
    text_location = (bbox.origin_x, bbox.origin_y - 10)
    cv2.putText(image_copy, result_text, text_location, cv2.FONT_HERSHEY_PLAIN, 1.5, (0, 255, 0), 2)

plt.figure(figsize=(10, 10))
plt.imshow(image_copy)
plt.axis('off')
plt.title('AccessApp: On-Device Obstacle Radar Verification')
plt.show()

## Part 2: Live ASL Translator (Hand Landmark Tracking)
This section verifies the **MediaPipe Handpose** model used for identifying 21 3D spatial hand landmarks. These landmarks are used in AccessApp to translate static ASL alphabetic gestures.

In [ ]:
# Download the Hand Landmarker task model
urllib.request.urlretrieve('https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task', 'hand_landmarker.task')

# Download an official Google MediaPipe test image of a hand
urllib.request.urlretrieve('https://storage.googleapis.com/mediapipe-tasks/hand_landmarker/woman_hands.jpg', 'woman_hands.jpg')

print("ASL Model and hand image downloaded successfully.")

In [ ]:
# Initialize the Hand Landmarker
base_options = mp.tasks.BaseOptions(model_asset_path='hand_landmarker.task')
options = mp.tasks.vision.HandLandmarkerOptions(base_options=base_options, num_hands=2)
hand_detector = mp.tasks.vision.HandLandmarker.create_from_options(options)

# Run Inference
hand_image = mp.Image.create_from_file('woman_hands.jpg')
hand_result = hand_detector.detect(hand_image)

# Visualize the 21 3D landmarks manually using OpenCV (guaranteed to work across versions)
annotated_image = np.copy(hand_image.numpy_view())
height, width, _ = annotated_image.shape

CONNECTIONS = [
    (0, 1), (1, 2), (2, 3), (3, 4),
    (0, 5), (5, 6), (6, 7), (7, 8),
    (5, 9), (9, 10), (10, 11), (11, 12),
    (9, 13), (13, 14), (14, 15), (15, 16),
    (13, 17), (17, 18), (18, 19), (19, 20),
    (0, 17)
]

for hand_landmarks in hand_result.hand_landmarks:
    # Draw connections
    for connection in CONNECTIONS:
        start_idx = connection[0]
        end_idx = connection[1]
        start_point = (int(hand_landmarks[start_idx].x * width), int(hand_landmarks[start_idx].y * height))
        end_point = (int(hand_landmarks[end_idx].x * width), int(hand_landmarks[end_idx].y * height))
        cv2.line(annotated_image, start_point, end_point, (0, 255, 0), 2)
        
    # Draw points
    for landmark in hand_landmarks:
        point = (int(landmark.x * width), int(landmark.y * height))
        cv2.circle(annotated_image, point, 4, (255, 0, 0), -1)

plt.figure(figsize=(10, 10))
plt.imshow(annotated_image)
plt.axis('off')
plt.title('AccessApp: On-Device ASL Skeleton Tracking Verification')
plt.show()

## Part 3: Why aren't the other two modules here?
If you are reviewing our codebase, you might notice that **AccessApp** features 4 core modules, but only 2 are verified in this notebook. Here is why:

*   **Notes-to-Audio (OCR):** This module uses **Google ML Kit Text Recognition V2**. ML Kit is a mobile-exclusive API natively tied to Google Play Services on Android/iOS. Because it is a native mobile library, it does not have a Python equivalent that can be executed in a cloud notebook environment like Colab.
*   **Color & Light Detector:** This module does not use a Neural Network. It relies on standard Computer Vision heuristics (extracting the central pixel matrix via Android `CameraX` and averaging the RGB/Luminance values mathematically). Because there is no ML model or `.tflite` file to run, there is nothing to verify in a Python ML Sandbox.